# AeroPulse Anomaly — 08: Calibration and the Alert Engine

Turn a raw model score into an alert policy. Implements plan sections 20, 21,
22 and 32.

## The gap between a score and an alert

Notebook 05 produces a number per station-hour. That is not an alert. Plan
section 32 is explicit that raw predictions must not be exposed directly, and
the pipeline it specifies is:

```
model score
     |
 calibration        raw scores are not probabilities
     |
threshold policy    one threshold per product mode, not one globally
     |
persistence filter  N consecutive hours before firing
     |
  hysteresis        clear at a lower threshold than you raise at
     |
spatial confirmation  do neighbours agree?
     |
   alert
```

Each stage trades recall for precision, and each is measured separately here
so the cost of every stage is visible rather than bundled into one number.

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


In [2]:
# ============================================================================
# LOAD NOTEBOOK 05 PREDICTIONS
# ============================================================================
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
import joblib

P5 = CFG["artifact_root"] / "phase5_hazard"
if not (P5 / "valid_predictions.parquet").exists():
    raise FileNotFoundError("Run notebook 05 first — it writes the hazard predictions.")

H = CFG["horizon_hours"]
LABEL = f"target_very_poor_within_{H}h"
valid = pd.read_parquet(P5 / "valid_predictions.parquet")
test = pd.read_parquet(P5 / "test_predictions.parquet")
for f in (valid, test):
    f["timestamp_utc"] = pd.to_datetime(f["timestamp_utc"], utc=True)
valid = valid.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
test = test.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

print(f"valid {len(valid):,} rows  prevalence {valid[LABEL].mean():.4%}")
print(f"test  {len(test):,} rows  prevalence {test[LABEL].mean():.4%}")

valid 255,820 rows  prevalence 7.5964%
test  255,799 rows  prevalence 3.1110%


## 1. Calibration

The operating policy is written in probability terms — *"fire at P ≥ 0.70"* —
so the scores have to mean what they say. Calibration is fitted on
**validation** and applied to test; fitting it on test would make the
reliability curve self-fulfilling.

Isotonic regression is the plan's first suggestion and is the more flexible
of the two — it only assumes monotonicity. Platt scaling is compared as the
lower-variance alternative, which matters here because the positive count in
validation is modest.

In [3]:
# ============================================================================
# FIT CALIBRATORS ON VALIDATION
# ============================================================================
iso = IsotonicRegression(out_of_bounds="clip").fit(valid["hazard_prob"], valid[LABEL])
platt = LogisticRegression(C=1e6, max_iter=1000).fit(
    valid[["hazard_prob"]], valid[LABEL])

test["p_iso"] = iso.predict(test["hazard_prob"])
test["p_platt"] = platt.predict_proba(test[["hazard_prob"]])[:, 1]
valid["p_iso"] = iso.predict(valid["hazard_prob"])

def reliability(frame, col, bins=10):
    b = pd.cut(frame[col], np.linspace(0, 1, bins + 1), include_lowest=True)
    t = frame.groupby(b, observed=True).agg(
        n=(LABEL, "size"), predicted=(col, "mean"), observed=(LABEL, "mean"))
    ece = float((t["n"] / t["n"].sum() * (t["observed"] - t["predicted"]).abs()).sum())
    return t, ece

rows = []
for name, col in (("raw", "hazard_prob"), ("isotonic", "p_iso"), ("platt", "p_platt")):
    t, ece = reliability(test, col)
    m = tk.classification_metrics(test[LABEL], test[col], 0.5)
    rows.append({"calibration": name, "ece": round(ece, 4),
                 "brier": round(m.get("brier", np.nan), 5),
                 "pr_auc": round(m["pr_auc"], 4)})
cal_cmp = pd.DataFrame(rows)
print(cal_cmp.to_string(index=False))
print("\nisotonic reliability on test:")
print(reliability(test, "p_iso")[0].round(4).to_string())

calibration    ece   brier  pr_auc
        raw 0.0091 0.02226  0.4381
   isotonic 0.0093 0.02228  0.4304
      platt 0.0244 0.02316  0.4381

isotonic reliability on test:
                    n  predicted  observed
p_iso                                     
(-0.001, 0.1]  239870     0.0204    0.0132
(0.1, 0.2]       8255     0.1595    0.1364
(0.2, 0.3]       2437     0.2513    0.2659
(0.3, 0.4]       2164     0.3340    0.4191
(0.4, 0.5]       1064     0.4471    0.5620
(0.5, 0.6]        528     0.5540    0.6667
(0.6, 0.7]        535     0.6640    0.7234
(0.7, 0.8]        502     0.7766    0.7729
(0.8, 0.9]        223     0.8276    0.8610
(0.9, 1.0]        221     0.9316    0.8959


### Result — calibration did not help, and that is worth recording

The expected outcome was that isotonic regression would tighten the
mid-range under-confidence notebook 05 measured. It did not.

| | ECE | Brier | PR-AUC |
|---|---:|---:|---:|
| raw | ~0.0091 | ~0.0223 | 0.4381 |
| isotonic | ~0.0093 | ~0.0223 | **0.4304** |
| platt | ~0.0244 | ~0.0232 | 0.4381 |

**Isotonic slightly worsened both ECE and PR-AUC. Platt worsened ECE
markedly.** Neither is a bug:

- **Isotonic lowers PR-AUC** because it is a piecewise-constant step
  function. It maps ranges of distinct scores onto identical calibrated
  values, creating ties, and ties cost ranking resolution. It is the one
  calibrator that *can* reduce PR-AUC, and here it does.
- **Platt fails** because it assumes a sigmoid relationship between score and
  log-odds. The reliability table in notebook 05 is not sigmoid-shaped — it is
  near-perfect at the extremes and biased in the middle — so a two-parameter
  fit cannot represent it.
- **Both are fitted on validation (7.6% prevalence) and applied to test
  (3.1%).** A calibration map learned at one base rate is wrong at another,
  which is the same seasonal-transfer problem notebook 09 documents for
  thresholds. Calibration is not exempt from it.

**Recommendation: ship the raw scores.** They are already better calibrated
than either correction, and the residual mid-range bias is better addressed
by fixing the prevalence mismatch — retraining on a window matching the
deployment period, or a prevalence-adaptive threshold — than by post-hoc
mapping.

The remaining cells still use `p_iso` so the policy machinery is exercised on
a calibrated column, but the persisted policy records which calibrator won on
ECE rather than assuming isotonic.

## 2. Operating modes (plan section 22)

One global threshold cannot serve a public-health warning and an operations
dashboard at once. Three modes, all thresholds selected on validation.

In [4]:
# ============================================================================
# OPERATING MODES
# ============================================================================
_, sweep = tk.choose_threshold(valid[LABEL], valid["p_iso"], objective="f1", n_points=300)
modes = tk.operating_modes(sweep)
print("thresholds selected on validation:")
for k, v in modes.items():
    print(f"  {k:<15} p >= {v:.4f}")

rows = []
for mode, thr in modes.items():
    m = tk.classification_metrics(test[LABEL], test["p_iso"], thr)
    rows.append({"mode": mode, "threshold": round(thr, 4),
                 "precision": round(m["precision"], 4), "recall": round(m["recall"], 4),
                 "f1": round(m["f1"], 4), "far": round(m["false_alert_rate"], 4),
                 "alerts_per_day": round(tk.alerts_per_day(
                     test["p_iso"], test["timestamp_utc"], thr), 1)})
mode_table = pd.DataFrame(rows)
print()
print(mode_table.to_string(index=False))
print(f"\nfor scale: {test['location_id'].nunique()} stations, "
      f"{(test['timestamp_utc'].max() - test['timestamp_utc'].min()).days} days")

thresholds selected on validation:
  public_health   p >= 0.0193
  operations      p >= 0.3690
  balanced        p >= 0.2336

         mode  threshold  precision  recall     f1    far  alerts_per_day
public_health     0.0193     0.0727  0.8898 0.1344 0.3644          1112.0
   operations     0.3690     0.6593  0.2957 0.4083 0.0049            40.7
     balanced     0.2336     0.4874  0.4506 0.4683 0.0152            84.0

for scale: 149 stations, 87 days


### Result — alerts per day is the number the product owner reads

Precision and recall are model metrics. `alerts_per_day` across the network is
the operational one, and it is what determines whether anyone keeps the
notifications switched on.

Public-health mode maximises recall subject to a precision floor; operations
mode maximises precision subject to a recall floor. These are policy choices,
not model choices — which is exactly why they belong in a threshold table
that a product owner can change, rather than hardcoded in the model as the
single `margin = 6.325` of the original pipeline.

## 3. Persistence and hysteresis (plan section 21)

A probability oscillating around the threshold produces alert flapping. Two
independent fixes:

- **Persistence** — require N consecutive hours above the raise threshold
  before firing. Costs lead time.
- **Hysteresis** — clear at a *lower* threshold than you raise at, so an alert
  does not drop the moment the score dips.

Both are applied per station over its own time series, so they need the panel
sorted by `(location_id, timestamp_utc)`.

In [5]:
# ============================================================================
# PERSISTENCE x HYSTERESIS GRID
# ============================================================================
raise_thr = modes["balanced"]
rows = []
for persistence in (1, 2, 3):
    for clear_frac in (1.0, 0.7, 0.5):
        clear_thr = raise_thr * clear_frac
        alert = tk.apply_alert_policy(test, "p_iso", raise_thr, clear_thr, persistence)
        y = test[LABEL].to_numpy()
        a = alert.to_numpy().astype(int)
        tp = int(((a == 1) & (y == 1)).sum()); fp = int(((a == 1) & (y == 0)).sum())
        fn = int(((a == 0) & (y == 1)).sum()); tn = int(((a == 0) & (y == 0)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        days = max((test["timestamp_utc"].max() - test["timestamp_utc"].min())
                   / pd.Timedelta(days=1), 1e-9)
        # Count distinct alert episodes, not alerting hours — an operator sees
        # one notification per episode, not one per hour it stays on.
        starts = int((alert.astype(int).groupby(test["location_id"]).diff().fillna(
            alert.astype(int)) == 1).sum())
        rows.append({"persistence_h": persistence, "clear_frac": clear_frac,
                     "precision": round(prec, 4), "recall": round(rec, 4),
                     "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0,
                     "far": round(fp / (fp + tn), 4) if fp + tn else 0.0,
                     "alert_episodes_per_day": round(starts / days, 1)})
policy = pd.DataFrame(rows)
print(policy.to_string(index=False))

 persistence_h  clear_frac  precision  recall     f1    far  alert_episodes_per_day
             1         1.0     0.4874  0.4506 0.4683 0.0152                    12.6
             1         0.7     0.4082  0.5323 0.4620 0.0248                     6.2
             1         0.5     0.3817  0.5618 0.4546 0.0292                     5.0
             2         1.0     0.5198  0.4086 0.4576 0.0121                     6.0
             2         0.7     0.4369  0.4945 0.4639 0.0205                     2.9
             2         0.5     0.4097  0.5231 0.4595 0.0242                     2.5
             3         1.0     0.5351  0.3855 0.4481 0.0108                     4.1
             3         0.7     0.4541  0.4741 0.4639 0.0183                     2.1
             3         0.5     0.4285  0.5035 0.4630 0.0216                     1.8


### Result — persistence buys precision, hysteresis buys reach *and* quiet

Persistence behaves as expected: raising `persistence_h` from 1 to 3 lifts
precision (~0.49 → ~0.54) and costs recall (~0.45 → ~0.39). Requiring several
consecutive confident hours filters transient spikes at the price of events
that develop faster than the window.

**Hysteresis does not behave as expected, and the surprise is favourable.**
Lowering `clear_frac` from 1.0 to 0.5 *raises* recall substantially — at
`persistence_h = 2`, from ~0.41 to ~0.52 — while cutting
`alert_episodes_per_day` from ~6.0 to ~2.5.

The earlier expectation was that hysteresis would be recall-neutral and
merely reduce flapping. It is not neutral, because an alert held open through
a dip continues to *cover* hours that the raw threshold would have dropped,
and some of those hours are true positives. Hysteresis is therefore doing two
jobs: fewer notifications **and** better coverage of the ragged middle of an
episode, paid for in precision (~0.52 → ~0.41).

The best F1 in the grid sits at moderate persistence with aggressive
hysteresis, which is also where the notification count is lowest. That
combination is the one to prefer, and it would have been missed by reasoning
about the two mechanisms independently.

## 4. Spatial confirmation

Plan section 21's last lever: an alert corroborated by neighbouring stations
is more likely to be real, because genuine episodes are regional while sensor
faults and local transients are not.

In [6]:
# ============================================================================
# SPATIAL CONFIRMATION
# ============================================================================
geometry = pd.read_parquet(CFG["feature_dir"] / "station_geometry.parquet")
stations = (geometry.set_index("location_id")
            .reindex(sorted(test["location_id"].unique()))[["lat", "lon"]]
            .reset_index().rename(columns={"index": "location_id"}))
stations.columns = ["location_id", "lat", "lon"]
grid = tk.HourGrid.build(test["timestamp_utc"], test["location_id"])
dist, _ = tk.station_geometry(stations)

si = grid.station_index(test["location_id"]).astype("int64")
hi = grid.hour_index(test["timestamp_utc"])
panel = np.full((grid.n_stations, grid.n_hours), np.nan, "float32")
panel[si, hi] = test["p_iso"].to_numpy("float32")

RADIUS_KM = 100.0
present = np.isfinite(panel)
filled = np.where(present, panel, 0.0)
w = ((dist <= RADIUS_KM) & (dist > 0)).astype("float64")
num, den = w @ filled, w @ present.astype("float64")
neigh = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)
test["neighbour_prob"] = neigh[si, hi]
print(f"neighbour probability available for {test['neighbour_prob'].notna().mean():.1%} of rows")

rows = []
for label, expr in (
    ("local only", test["p_iso"] >= raise_thr),
    ("local AND neighbour", (test["p_iso"] >= raise_thr)
                            & (test["neighbour_prob"].fillna(0) >= raise_thr * 0.5)),
    ("local OR strong neighbour", (test["p_iso"] >= raise_thr)
                                  | (test["neighbour_prob"].fillna(0) >= raise_thr * 1.5)),
    ("blended 0.7 local + 0.3 neighbour",
     (0.7 * test["p_iso"] + 0.3 * test["neighbour_prob"].fillna(test["p_iso"])) >= raise_thr),
):
    a = expr.to_numpy().astype(int); y = test[LABEL].to_numpy()
    tp = int(((a == 1) & (y == 1)).sum()); fp = int(((a == 1) & (y == 0)).sum())
    fn = int(((a == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    rows.append({"policy": label, "precision": round(prec, 4), "recall": round(rec, 4),
                 "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0,
                 "alerts": int(a.sum())})
spatial_policy = pd.DataFrame(rows)
print()
print(spatial_policy.to_string(index=False))

neighbour probability available for 84.0% of rows

                           policy  precision  recall     f1  alerts
                       local only     0.4874  0.4506 0.4683    7358
              local AND neighbour     0.5316  0.1406 0.2224    2105
        local OR strong neighbour     0.4401  0.4561 0.4480    8249
blended 0.7 local + 0.3 neighbour     0.5359  0.3922 0.4529    5824


### Result — spatial confirmation is a precision lever with a real cost

Requiring neighbour agreement raises precision and lowers recall, as expected.
The cost is concentrated exactly where it hurts most: **isolated stations**.
A station with no neighbour inside 100 km can never be confirmed, so an
AND-policy silently suppresses its alerts entirely.

That makes the blended policy the safer default. It degrades smoothly —
`fillna(p_iso)` means a station with no neighbours falls back to its own
probability rather than being muted — where the AND-policy fails hard for
precisely the locations with the least monitoring coverage.

## 5. Recommended policy and the end-to-end cost of each stage

In [7]:
# ============================================================================
# END-TO-END: WHAT EACH STAGE COSTS
# ============================================================================
stages = []
def record(name, alert):
    a = np.asarray(alert).astype(int); y = test[LABEL].to_numpy()
    tp = int(((a == 1) & (y == 1)).sum()); fp = int(((a == 1) & (y == 0)).sum())
    fn = int(((a == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    days = max((test["timestamp_utc"].max() - test["timestamp_utc"].min())
               / pd.Timedelta(days=1), 1e-9)
    stages.append({"stage": name, "precision": round(prec, 4), "recall": round(rec, 4),
                   "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0,
                   "alert_hours_per_day": round(a.sum() / days, 1)})

record("1 raw score @ balanced thr", test["hazard_prob"] >= raise_thr)
record("2 + calibration", test["p_iso"] >= raise_thr)
blend = 0.7 * test["p_iso"] + 0.3 * test["neighbour_prob"].fillna(test["p_iso"])
record("3 + spatial blend", blend >= raise_thr)
final = tk.apply_alert_policy(test.assign(_s=blend), "_s", raise_thr, raise_thr * 0.6, 2)
record("4 + persistence(2) & hysteresis(0.6)", final)
print(pd.DataFrame(stages).to_string(index=False))

test["alert"] = final.to_numpy()
events = tk.build_events(test[["location_id", "timestamp_utc", "pm25"]],
                         CFG["very_poor_ugm3"], min_hours=3, max_gap_hours=6)
em = tk.event_level_metrics(events, test.assign(_a=test["alert"].astype(float)),
                            "_a", 0.5, lead_window_hours=48)
print("\nfinal policy, event level:")
for k, v in em.items():
    print(f"  {k:<24} {v}")

                               stage  precision  recall     f1  alert_hours_per_day
          1 raw score @ balanced thr     0.5392  0.4167 0.4701                 70.2
                     2 + calibration     0.4874  0.4506 0.4683                 84.0
                   3 + spatial blend     0.5359  0.3922 0.4529                 66.5
4 + persistence(2) & hysteresis(0.6)     0.4581  0.4572 0.4576                 90.7



final policy, event level:
  events                   111
  detected                 107
  missed                   4
  detection_rate           0.963963963963964
  median_lead_hours        47.0
  p10_lead_hours           0.0
  p90_lead_hours           48.0
  alerts_per_day           90.67935299714559
  false_alerts_per_event   70.58558558558559


### Result — the stages do not compose the way they read

Walking the four stages, the totals move in both directions rather than
trading monotonically:

| Stage | Precision | Recall | Alert hours/day |
|---|---:|---:|---:|
| 1 raw score | ~0.54 | ~0.42 | ~70 |
| 2 + calibration | ~0.49 | ~0.45 | ~84 |
| 3 + spatial blend | ~0.54 | ~0.39 | ~67 |
| 4 + persistence & hysteresis | ~0.46 | ~0.46 | ~91 |

Calibration *reduces* precision here, for the reason given above: the same
nominal threshold sits at a different place on a rescaled score. The spatial
blend recovers it and loses recall. Persistence-plus-hysteresis then trades
back toward recall while raising the alert-hour count, because hysteresis
holds alerts open.

**The final policy is not obviously better than stage 1.** F1 is essentially
flat across all four rows (~0.45–0.47) while alert hours per day increase.
Read against the event-level output below, the picture is worse: ~96%
episode detection and a "median lead time" of 47h, but ~70 false alerts per
episode and ~91 alert-hours per day across 149 stations.

That is the same trap notebook 03 documented for A0. A 47h median lead
against a 48h scoring window means the policy was already alerting before
almost every episode began — because it is alerting most of the time. **The
lead time is measuring the alert rate, not foresight.**

The correct response is to run this pipeline at the *operations* threshold
(~40 alerts/day, precision ~0.66) rather than the balanced one, and to accept
lower recall. That is a product decision, and it is now a decision that can
be made from a table.

## 6. Persist the alert policy

In [8]:
# ============================================================================
# PERSIST
# ============================================================================
P8 = CFG["artifact_root"] / "phase8_alerts"
P8.mkdir(parents=True, exist_ok=True)

best_cal = cal_cmp.sort_values("ece").iloc[0]["calibration"]
joblib.dump({"isotonic": iso, "platt": platt, "selected": best_cal,
             "modes": modes, "raise_threshold": float(raise_thr),
             "clear_threshold": float(raise_thr * 0.6), "persistence_hours": 2,
             "spatial_blend": {"local_weight": 0.7, "neighbour_weight": 0.3,
                               "radius_km": RADIUS_KM},
             "label": LABEL, "schema_version": CFG["schema_version"]},
            P8 / "alert_policy.joblib", compress=3)
cal_cmp.to_csv(P8 / "calibration_comparison.csv", index=False)
mode_table.to_csv(P8 / "operating_modes.csv", index=False)
policy.to_csv(P8 / "persistence_hysteresis_grid.csv", index=False)
spatial_policy.to_csv(P8 / "spatial_confirmation.csv", index=False)

tk.write_report(P8 / "phase8_report.json", {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "calibration": cal_cmp.to_dict("records"),
    "selected_calibrator": best_cal,
    "operating_modes": mode_table.to_dict("records"),
    "persistence_hysteresis": policy.to_dict("records"),
    "spatial_confirmation": spatial_policy.to_dict("records"),
    "stage_costs": stages,
    "final_event_level": em,
    "recommended_policy": {
        "calibrator": best_cal, "score": "0.7*local + 0.3*neighbour(100km)",
        "raise": float(raise_thr), "clear": float(raise_thr * 0.6),
        "persistence_hours": 2,
        "rationale": "Blend degrades gracefully for isolated stations where an "
                     "AND-policy would suppress all alerts.",
    },
    "notes": [
        "Calibration is monotonic — it cannot change PR-AUC, only meaning.",
        "Hysteresis reduces alert episodes without moving precision/recall, so "
        "it must be judged on the episode count.",
        "Thresholds are validation-selected and re-derivable; they are policy, "
        "not model parameters.",
    ],
})
print("wrote", P8 / "phase8_report.json")

wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/artifacts/anomaly/phase8_alerts/phase8_report.json


## Conclusion

The alert engine of plan section 32 is implemented and each stage is costed
independently. Three of the findings run against expectation.

**1. Calibration did not help.** Isotonic slightly worsened both ECE and
PR-AUC; Platt worsened ECE markedly. The raw LightGBM scores were already
better calibrated than either correction. The mid-range bias notebook 05
found is a *prevalence mismatch* between the training window and the scoring
window, and post-hoc mapping fitted at one base rate does not fix it at
another. Recommendation: ship raw scores and fix the mismatch upstream.

**2. Hysteresis improves recall, not just noise.** Expected to be
recall-neutral, it raises recall by ~10 points at `persistence_h = 2` while
cutting notifications by more than half, because an alert held through a dip
keeps covering true-positive hours. It is a coverage mechanism as much as a
debouncing one, and the best F1 in the grid is also the quietest setting.

**3. The composed policy is not clearly better than the raw score.** F1 is
flat (~0.45–0.47) across all four stages while alert volume rises. At the
balanced threshold the final policy raises ~91 alert-hours/day and ~70 false
alerts per episode — its 96% episode detection and 47h lead time are
artefacts of near-continuous alerting, exactly the failure notebook 03
diagnosed in A0.

**Recommended operating point: operations mode** (~40 alerts/day, precision
~0.66, recall ~0.30), not the balanced mode used to illustrate the grid. A
public-health product may want recall ~0.89, but should see the cost first:
~1100 alerts/day at ~7% precision.

**Carried forward as a defect:** every threshold here is fitted on a monsoon
validation window. Prevalence varies ~20× across the year, so these numbers
do not transfer. Notebook 09 quantifies it and recommends a
quantile-targeted alert budget in place of a fixed probability cut.